# Real-Time Multimodal Voice Assistant: Tutorial Notebook

This notebook is a step-by-step tutorial version of the project.
It uses the same production-style pipeline code from `src/realtime_mm` and runs it end-to-end.

What you will learn:
- how the latency budget is decomposed,
- how to run normal vs stress scenarios,
- how graceful degradation appears in metrics,
- how to validate behavior with tests.

## Step 1: Setup Imports and Path

We keep the existing project untouched and only import from `src/realtime_mm`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Using src path: {src_path}")

Project root: /home/ahmad/AI/Projects/realtime-multimodal-voice-assistant
Using src path: /home/ahmad/AI/Projects/realtime-multimodal-voice-assistant/src


## Step 2: Import the Pipeline Components

In [2]:
from realtime_mm.config import PipelineSettings
from realtime_mm.demo_inputs import get_demo_utterances
from realtime_mm.pipeline import RealTimePipeline

## Step 3: Understand the Configured Latency Budget

This table is the system's target budget decomposition.
It includes stage-level budgets and queueing slack reserve.

In [3]:
settings_normal = PipelineSettings(profile="normal", request_count=8, random_seed=7)
pipeline_normal = RealTimePipeline(settings=settings_normal)

print(pipeline_normal.report.render_budget_table(settings_normal))

Stage          | Target (ms) | Timeout (ms) | Notes                                 
---------------+-------------+--------------+---------------------------------------
ingress        | 180         | 260          | capture + VAD + enqueue               
asr            | 280         | 380          | speech-to-text                        
llm            | 520         | 760          | response generation                   
tts            | 260         | 420          | text-to-speech                        
output         | 90          | 140          | stream/send result                    
queueing_slack | 70          | -            | queue wait + scheduling jitter reserve
end_to_end     | 1400        | -            | global deadline                       


## Step 4: Run a Normal-Load Streaming Session

We run 8 requests through the async pipeline and observe end-to-end behavior.

In [4]:
utterances_normal = get_demo_utterances(8)
normal_results = await pipeline_normal.run(utterances_normal)

print(f"Completed requests: {len(normal_results)}")
print(pipeline_normal.report.render_runtime_summary())

2026-06-12 08:29:49.775 | INFO     | realtime_mm.pipeline:_finalize:430 - req-001 | mode=normal | e2e=806.8ms | deadline=OK | audio=True | degraded=none


2026-06-12 08:29:50.112 | INFO     | realtime_mm.pipeline:_finalize:430 - req-002 | mode=normal | e2e=983.6ms | deadline=OK | audio=True | degraded=none


2026-06-12 08:29:50.481 | INFO     | realtime_mm.pipeline:_finalize:430 - req-003 | mode=normal | e2e=1223.2ms | deadline=OK | audio=True | degraded=none


2026-06-12 08:29:50.626 | INFO     | realtime_mm.pipeline:_finalize:430 - req-004 | mode=normal | e2e=1217.3ms | deadline=OK | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:50.940 | INFO     | realtime_mm.pipeline:_finalize:430 - req-005 | mode=normal | e2e=1378.4ms | deadline=OK | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:51.247 | INFO     | realtime_mm.pipeline:_finalize:430 - req-006 | mode=normal | e2e=1557.3ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:51.628 | INFO     | realtime_mm.pipeline:_finalize:430 - req-007 | mode=normal | e2e=1798.2ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:51.980 | INFO     | realtime_mm.pipeline:_finalize:430 - req-008 | mode=normal | e2e=1999.9ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


Completed requests: 8
Metric            | N | P50 ms | P95 ms | Timeout/Miss | Errors | Skipped/Degraded | Dropped
------------------+---+--------+--------+--------------+--------+------------------+--------
ingress           | 8 | 150.7  | 161.8  | 0            | 0      | 0                | 0      
asr               | 8 | 159.1  | 246.3  | 0            | 0      | 0                | 0      
llm               | 8 | 331.3  | 371.8  | 0            | 0      | 0                | 0      
tts               | 8 | 0.0    | 174.4  | 0            | 0      | 5                | 0      
output            | 8 | 43.6   | 58.4   | 0            | 0      | 0                | 0      
queueing_overhead | 8 | 636.3  | 1201.2 | -            | -      | -                | -      
end_to_end        | 8 | 1300.8 | 1929.3 | 3            | -      | 5                | -      


## Step 5: Inspect One Request in Detail

In [5]:
sample = normal_results[0]

print("Request ID:", sample.request_id)
print("Mode at ingress:", sample.mode_at_ingress)
print("End-to-end ms:", round(sample.end_to_end_ms, 2))
print("Deadline met:", sample.deadline_met)
print("Degraded reasons:", sample.degraded_reasons or ["none"])
print("Audio rendered:", sample.audio_rendered)
print("Transcript:", sample.transcript)
print("Reply:", sample.reply_text)
print("\nStage results:")
for stage_name in ["ingress", "asr", "llm", "tts", "output"]:
    sr = sample.stage_results.get(stage_name)
    if sr is None:
        continue
    print(
        f"- {stage_name}: status={sr.status}, latency={sr.latency_ms:.1f} ms, "
        f"target={sr.target_ms} ms, timeout={sr.timeout_ms} ms"
    )

Request ID: req-001
Mode at ingress: normal
End-to-end ms: 806.85
Deadline met: True
Degraded reasons: ['none']
Audio rendered: True
Transcript: What's the weather in Mumbai this evening and should I carry an umbrella?
Reply: Actionable summary: What's the weather in Mumbai this evening and should I carry an umbrella?. Recommended next step: confirm intent and execute safely.

Stage results:
- ingress: status=ok, latency=160.0 ms, target=180 ms, timeout=260 ms
- asr: status=ok, latency=193.8 ms, target=280 ms, timeout=380 ms
- llm: status=ok, latency=247.4 ms, target=520 ms, timeout=760 ms
- tts: status=ok, latency=162.3 ms, target=260 ms, timeout=420 ms
- output: status=ok, latency=42.0 ms, target=90 ms, timeout=140 ms


## Step 6: Run a Stress Scenario

The stress profile increases latency spikes.
You should see more deadline misses/timeouts and stronger degradation behavior.

In [6]:
settings_stress = PipelineSettings(profile="stress", request_count=8, random_seed=7)
pipeline_stress = RealTimePipeline(settings=settings_stress)

utterances_stress = get_demo_utterances(8)
stress_results = await pipeline_stress.run(utterances_stress)

print(f"Completed requests: {len(stress_results)}")
print(pipeline_stress.report.render_runtime_summary())

2026-06-12 08:29:53.327 | INFO     | realtime_mm.pipeline:_finalize:430 - req-001 | mode=normal | e2e=1313.1ms | deadline=OK | audio=False | degraded=asr_timeout_fallback,tts_skipped_for_latency


2026-06-12 08:29:53.835 | INFO     | realtime_mm.pipeline:_finalize:430 - req-002 | mode=normal | e2e=1662.1ms | deadline=MISS | audio=False | degraded=asr_timeout_fallback,tts_skipped_for_latency


2026-06-12 08:29:54.456 | INFO     | realtime_mm.pipeline:_finalize:430 - req-003 | mode=normal | e2e=2153.3ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:55.123 | INFO     | realtime_mm.pipeline:_finalize:430 - req-004 | mode=normal | e2e=2679.3ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


2026-06-12 08:29:55.863 | INFO     | realtime_mm.pipeline:_finalize:430 - req-005 | mode=normal | e2e=3267.2ms | deadline=MISS | audio=False | degraded=llm_timeout_fallback,tts_skipped_for_latency


2026-06-12 08:29:56.620 | INFO     | realtime_mm.pipeline:_finalize:430 - req-006 | mode=normal | e2e=3857.7ms | deadline=MISS | audio=False | degraded=asr_timeout_fallback,llm_timeout_fallback,tts_skipped_for_latency


2026-06-12 08:29:57.431 | INFO     | realtime_mm.pipeline:_finalize:430 - req-007 | mode=normal | e2e=4555.6ms | deadline=MISS | audio=False | degraded=llm_timeout_fallback,tts_skipped_for_latency


2026-06-12 08:29:58.135 | INFO     | realtime_mm.pipeline:_finalize:430 - req-008 | mode=normal | e2e=5113.2ms | deadline=MISS | audio=False | degraded=tts_skipped_for_latency


Completed requests: 8
Metric            | N | P50 ms | P95 ms | Timeout/Miss | Errors | Skipped/Degraded | Dropped
------------------+---+--------+--------+--------------+--------+------------------+--------
ingress           | 8 | 149.1  | 163.9  | 0            | 0      | 0                | 0      
asr               | 8 | 266.3  | 381.6  | 3            | 0      | 0                | 0      
llm               | 8 | 714.8  | 763.0  | 3            | 0      | 0                | 0      
tts               | 8 | 0.0    | 0.0    | 0            | 0      | 8                | 0      
output            | 8 | 66.9   | 105.4  | 0            | 0      | 0                | 0      
queueing_overhead | 8 | 1825.2 | 3710.2 | -            | -      | -                | -      
end_to_end        | 8 | 2973.2 | 4918.0 | 7            | -      | 8                | -      


## Step 7: Compare Normal vs Stress End-to-End Metrics

In [7]:
normal_e2e = pipeline_normal.report.end_to_end_summary()
stress_e2e = pipeline_stress.report.end_to_end_summary()

print("Normal profile:")
for key, value in normal_e2e.items():
    print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

print("\nStress profile:")
for key, value in stress_e2e.items():
    print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

Normal profile:
  count: 8.000
  p50_ms: 1300.820
  p95_ms: 1929.283
  mean_ms: 1370.604
  deadline_miss_count: 3.000
  deadline_miss_ratio: 0.375
  degraded_count: 5.000

Stress profile:
  count: 8.000
  p50_ms: 2973.228
  p95_ms: 4918.027
  mean_ms: 3075.172
  deadline_miss_count: 7.000
  deadline_miss_ratio: 0.875
  degraded_count: 8.000


## Step 8: Run Automated Tests from Notebook

This confirms fallback and deadline behavior with deterministic test cases.

In [8]:
import subprocess

result = subprocess.run(
    ["uv", "run", "pytest", "-q"],
    cwd=project_root,
    text=True,
    capture_output=True,
    check=False,
)

print(result.stdout)
if result.stderr.strip():
    print("stderr:")
    print(result.stderr)

print("Return code:", result.returncode)

...                                                                      [100%]
3 passed in 1.08s

stderr:

Return code: 0


## Step 9: Next Learning Exercises

1. Change `end_to_end_budget_ms` and observe deadline miss changes.
2. Tighten `llm.timeout_ms` to force more fallback responses.
3. Increase request count to see queueing overhead grow.
4. Replace simulated services with real ASR/LLM/TTS providers while keeping timeout wrappers.